In [7]:
import json, re

YEAR = re.compile(r"\b(20\d{2}|19\d{2})\b") 

def year_of(publish_date):
    m = YEAR.search(publish_date or "")
    return int(m.group()) if m else None

keep = {}          # work_key -> edition fields we want
with open("../data/open_library/ol_dump_editions_2026-05-31.txt", "rt") as f:
    for line in f:
        rec = json.loads(line.split("\t", 4)[4])
        yr = year_of(rec.get("publish_date"))
        isbns = rec.get("isbn_13")
        works = rec.get("works")
        if yr and yr >= 2018 and isbns and works:
            keep[works[0]["key"]] = {
                "isbn13": isbns[0],
                "title": rec.get("title"),
                "subtitle": rec.get("subtitle"),
                "published_year": yr,
                "num_pages": rec.get("number_of_pages"),
                "cover_id": (rec.get("covers") or [None])[0],
                # editions reference author *keys* (/authors/OL...A), resolved later
                "author_keys": [a["key"] for a in rec.get("authors", []) if "key" in a],
            }

In [8]:
# descriptions and subjects live on the work, not the edition, so stream the
# works dump and fill them into the rows pass 1 kept. The works dump is the big one
# so we check `key in keep` (a cheap string lookup) BEFORE json.loads —
# only the handful of works we actually want get parsed.

def description_text(work):
    d = work.get("description")
    if isinstance(d, dict):          # {"type": "/type/text", "value": "..."}
        d = d.get("value")
    return d or None

def work_author_keys(work):
    # work authors are usually [{"author": {"key": ...}}], older records use [{"key": ...}]
    keys = []
    for a in work.get("authors", []):
        if not isinstance(a, dict):
            continue
        if isinstance(a.get("author"), dict) and "key" in a["author"]:
            keys.append(a["author"]["key"])
        elif "key" in a:
            keys.append(a["key"])
    return keys

found = 0
with open("../data/open_library/ol_dump_works_2026-05-31.txt", "rt") as f:
    for line in f:
        cols = line.split("\t", 4)
        key = cols[1]                # column 2 is the work key, e.g. /works/OL...W
        if key not in keep:
            continue
        work = json.loads(cols[4])
        keep[key]["description"] = description_text(work)
        keep[key]["categories"] = ";".join(work.get("subjects", [])) or None
        if not keep[key]["author_keys"]:       # edition had no authors; fall back to work
            keep[key]["author_keys"] = work_author_keys(work)
        found += 1

print(f"works matched:      {found:,} / {len(keep):,} editions kept")
have_desc = sum(1 for v in keep.values() if v.get("description"))
print(f"with a description: {have_desc:,} ({have_desc / len(keep):.0%})")

works matched:      7,220,536 / 7,220,586 editions kept
with a description: 159,424 (2%)


In [9]:
# resolve author names. We only have author keys so far; the names live in the
# authors dump. Collect the distinct keys we need, then stream the authors dump once,
# parsing only the records whose key we want.

needed_authors = {k for v in keep.values() for k in v["author_keys"]}
print("distinct author keys needed:", f"{len(needed_authors):,}")

author_names = {}
with open("../data/open_library/ol_dump_authors_2026-05-31.txt", "rt") as f:
    for line in f:
        cols = line.split("\t", 4)
        key = cols[1]                # column 2 is the author key, e.g. /authors/OL...A
        if key not in needed_authors:
            continue
        name = json.loads(cols[4]).get("name")
        if name:
            author_names[key] = name

print("names resolved:", f"{len(author_names):,}")

distinct author keys needed: 3,519,129
names resolved: 3,518,932


In [10]:
# assemble the kept rows into the books_cleaned.csv schema, resolving the author names
# collected above, run the same cleaning as the rest of the pipeline, and write the CSV
# the downstream notebooks read.

import numpy as np
import pandas as pd

COVER_URL = "https://covers.openlibrary.org/b/id/{}-L.jpg"

books = pd.DataFrame(keep.values())
print("rows from dumps:", len(books))

# resolve author keys to names; OL dumps carry no ratings, so average_rating stays blank.
books["authors"] = books["author_keys"].apply(
    lambda ks: ";".join(author_names[k] for k in ks if k in author_names) or None
)
books["average_rating"] = np.nan

# thumbnail comes from the cover id
books["thumbnail"] = books["cover_id"].apply(
    lambda c: COVER_URL.format(int(c)) if pd.notna(c) else None
)

# same cleaning as the Google Books pipeline: need an isbn13 + a description, keep only
# descriptions with >= 20 words, and one row per isbn13.
books = books.dropna(subset=["isbn13", "description"])
books = books[books["description"].str.split().str.len() >= 20]
books = books.drop_duplicates(subset="isbn13").reset_index(drop=True)
print("after cleaning + dedupe:", len(books))

# pair title with subtitle when present
books["title_and_subtitle"] = np.where(
    books["subtitle"].isna(),
    books["title"],
    books["title"] + ": " + books["subtitle"],
)

# isbn13-prefixed description is the join key between the vector store and the dataframe
books["tagged_description"] = books[["isbn13", "description"]].astype(str).agg(" ".join, axis=1)

# match the exact column set the downstream pipeline expects
books = books[[
    "isbn13", "title", "authors", "categories", "description", "thumbnail",
    "published_year", "average_rating", "num_pages", "title_and_subtitle",
    "tagged_description",
]]

books.to_csv("../data/books_cleaned.csv", index=False)
print("wrote ../data/books_cleaned.csv")
books.head()

rows from dumps: 7220586
after cleaning + dedupe: 142889
wrote ../data/books_cleaned.csv


,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description
0,9781411668980,"The Galaxii Series: Book 1 ""Blachart""",Christina Engela,"Fiction, romance, general",Action! Adventure! Space Opera!\r\n\r\nLife ha...,https://covers.openlibrary.org/b/id/14313750-L...,2018,NaN,280.0,"The Galaxii Series: Book 1 ""Blachart""",9781411668980 Action! Adventure! Space Opera!\...
1,9781484785539,Brooklyn House magician's manual,Rick Riordan,Serie:The_Kane_Chronicles;Juvenile fiction;Egy...,Calling all young magicians of Egypt! If the b...,https://covers.openlibrary.org/b/id/14601514-L...,2018,NaN,185.0,Brooklyn House magician's manual: your guide t...,9781484785539 Calling all young magicians of E...
2,9780226575087,Lost Mars,Michael Ashley,"Fiction;Science Fiction;Fiction, science ficti...",Ten short stories from the golden age of scien...,https://covers.openlibrary.org/b/id/13133252-L...,2018,NaN,302.0,Lost Mars: stories from the golden age of the ...,9780226575087 Ten short stories from the golde...
3,9780062467874,Villain,Michael Grant,Juvenile fiction;Fiction;Supernatural;Horror s...,**MONSTER. VILLAIN. HERO.**\r\nWHICH SUPERCREA...,https://covers.openlibrary.org/b/id/8814378-L.jpg,2018,NaN,324.0,Villain,9780062467874 **MONSTER. VILLAIN. HERO.**\r\nW...
4,9788467066357,El misterio de la guía de ferrocarriles,Agatha Christie;José Mallorquí Figuerola,Fiction;Mystery;Agatha Christie;Hercule Poirot...,"There's a serial killer on the loose, bent on ...",https://covers.openlibrary.org/b/id/14286064-L...,2022,NaN,272.0,El misterio de la guía de ferrocarriles,9788467066357 There's a serial killer on the l...
